# Can a small model pass a test?

This notebook asks a fun question: **can a small browser model answer SAT-style multiple-choice questions?**

The point is not to prove that the model is brilliant. In fact, small browser models will often do poorly. That is part of the lesson. You will see where they make reasonable guesses, where they get confused, and how scoring works.

## Step 1: Get set up

This loads WebLLM, checks for WebGPU, and defines helpers for loading a chat model and asking questions in a consistent format.

In [ ]:
import js
import random
import re
import pandas as pd
from pyodide.ffi import to_js, create_proxy
from IPython.display import display

if getattr(js.navigator, "gpu", None):
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

engine = None
loaded_model = None


def js_obj(d):
    return to_js(d, dict_converter=js.Object.fromEntries)


def progress_bar():
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    return status, create_proxy(show)


async def start_model(model_id):
    global engine
    status, progress = progress_bar()

    async def attempt(chat_options=None):
        global engine
        if engine is None:
            if chat_options:
                engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress}), js_obj(chat_options)
                )
            else:
                engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress})
                )
        else:
            engine.setInitProgressCallback(progress)
            if chat_options:
                await engine.reload(model_id, js_obj(chat_options))
            else:
                await engine.reload(model_id)

    try:
        await attempt()
    except Exception as error:
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await attempt({"sliding_window_size": -1})
        else:
            raise


def build_prompt(question_text, choices_text, passage_text=""):
    if passage_text:
        body = "Passage: " + passage_text + "\n\nQuestion: " + question_text
    else:
        body = "Question: " + question_text
    return (
        "Here is an SAT-style multiple-choice question.\n\n"
        + body
        + "\n\nChoices: " + choices_text + "\n\n"
        + "Think carefully. On the very last line of your response, write exactly: Final answer: X where X is A, B, C, or D."
    )


async def answer_question(question_text, choices_text, passage_text=""):
    if loaded_model is None:
        return {"error": "No model is loaded yet. Run Step 3."}
    messages = [
        {"role": "system", "content": "You are a careful tutor. Show your reasoning, then end with exactly one line in the form Final answer: X."},
        {"role": "user", "content": build_prompt(question_text, choices_text, passage_text)},
    ]
    request = {
        "messages": messages,
        "temperature": 0.0,
        "seed": 7,
        "max_tokens": 300,
        "frequency_penalty": 0.5,
    }
    reply = await engine.chat.completions.create(js_obj(request))
    result = reply.to_py()
    text = result['choices'][0]['message']['content']
    match = re.search(r'[Ff]inal answer:\s*([A-D])', text)
    result['parsed_answer'] = match.group(1).upper() if match else ''
    return result

## Step 2: Choose and load a model

A small model will make the experiment more honest. It may score poorly, but that is the point.

In [ ]:
MODEL_ID = "gemma3-1b-it-q4f16_1-MLC"
print("Model to load: " + MODEL_ID)

In [ ]:
await start_model(MODEL_ID)
loaded_model = MODEL_ID
print(loaded_model + " is loaded and ready.")

## Step 3: Warm up with one question

We start with one simple question so you can see the prompt format and the answer parser.

In [ ]:
warmup_question = "If 3x + 5 = 14, what is the value of x?"
warmup_choices = "A) 2\nB) 3\nC) 4\nD) 5"

warmup = await answer_question(warmup_question, warmup_choices)
if 'error' in warmup:
    print(warmup['error'])
else:
    print(warmup['choices'][0]['message']['content'])
    print()
    print("Parsed answer:", warmup['parsed_answer'])

## Step 4: Build a small SAT-style question bank

To keep this notebook reliable in the browser, we start with a small built-in question bank written for this notebook. Later there is an optional step to try pulling a larger live bank from PineSAT.

In [ ]:
QUESTION_BANK = [
    {
        'section': 'Math',
        'difficulty': 'Easy',
        'question': 'If 2x + 7 = 19, what is the value of x?',
        'choices': 'A) 5\nB) 6\nC) 7\nD) 8',
        'answer': 'B',
        'passage': '',
    },
    {
        'section': 'Math',
        'difficulty': 'Medium',
        'question': 'A rectangle has length 9 and width 4. What is its area?',
        'choices': 'A) 13\nB) 18\nC) 26\nD) 36',
        'answer': 'D',
        'passage': '',
    },
    {
        'section': 'Math',
        'difficulty': 'Hard',
        'question': 'If y = 3(x - 2) + 4, what is y when x = 5?',
        'choices': 'A) 9\nB) 11\nC) 13\nD) 15',
        'answer': 'C',
        'passage': '',
    },
    {
        'section': 'English',
        'difficulty': 'Easy',
        'question': 'Which choice best completes the sentence so that it is grammatically correct? The students ___ excited for the field trip.',
        'choices': 'A) is\nB) are\nC) was\nD) be',
        'answer': 'B',
        'passage': '',
    },
    {
        'section': 'English',
        'difficulty': 'Medium',
        'question': 'Based on the passage, why did Lena bring a notebook to the park?',
        'choices': 'A) She wanted to draw birds.\nB) She needed to solve math homework.\nC) She planned to write down story ideas.\nD) She was keeping score during a game.',
        'answer': 'C',
        'passage': 'Lena liked to sit on the same bench every Saturday morning. She watched joggers pass, listened to dogs bark at squirrels, and wrote down the bits of conversation she overheard. By noon she usually had three new ideas for short stories.',
    },
    {
        'section': 'English',
        'difficulty': 'Hard',
        'question': 'Which choice best states the main idea of the passage?',
        'choices': 'A) City buses should run later at night.\nB) A neighborhood garden changed how neighbors interacted.\nC) Growing tomatoes is harder than people expect.\nD) Children prefer parks to gardens.',
        'answer': 'B',
        'passage': 'When the empty lot was finally cleared, most people expected another apartment building. Instead, volunteers built raised beds, painted signs, and invited anyone on the block to help. Within months, people who had barely nodded to one another were trading peppers, recipes, and advice about soil.',
    },
]

pd.DataFrame(QUESTION_BANK)[['section', 'difficulty', 'question', 'answer']]

## Step 5: Choose a section, difficulty, and set size

Because the models run on your machine, it is better to start small.

In [ ]:
SECTION = "English"
DIFFICULTY = "Easy"
NUM_QUESTIONS = 2

print("Section:    " + SECTION)
print("Difficulty: " + DIFFICULTY)
print("Questions:  " + str(NUM_QUESTIONS))

## Step 6: Build the practice set

This filters the question bank and picks a small sample.

In [ ]:
filtered = [
    item for item in QUESTION_BANK
    if item['section'] == SECTION and item['difficulty'] == DIFFICULTY
]

if not filtered:
    print("No questions match those settings. Change SECTION or DIFFICULTY.")
    practice_set = []
else:
    practice_set = filtered[:]
    random.shuffle(practice_set)
    practice_set = practice_set[:NUM_QUESTIONS]
    print("Built a practice set of", len(practice_set), "questions.")
    for i, item in enumerate(practice_set, start=1):
        print()
        print("Q" + str(i) + ":", item['question'])
        if item['passage']:
            print("Passage:", item['passage'])
        print("Choices:", item['choices'])
        print("Correct answer:", item['answer'])

## Step 7: Let the model take the mini test

The next cell asks each question, parses the final answer letter, and saves the results.

In [ ]:
results = []
for item in practice_set:
    reply = await answer_question(item['question'], item['choices'], item['passage'])
    if 'error' in reply:
        print(reply['error'])
        break
    model_text = reply['choices'][0]['message']['content']
    model_guess = reply['parsed_answer']
    is_correct = model_guess == item['answer']
    results.append({
        'section': item['section'],
        'difficulty': item['difficulty'],
        'question': item['question'],
        'correct_answer': item['answer'],
        'model_guess': model_guess,
        'correct': is_correct,
        'raw_reply': model_text,
    })

results_table = pd.DataFrame(results)
results_table[['section', 'difficulty', 'correct_answer', 'model_guess', 'correct']]

## Step 8: Read the model's full replies

Sometimes the most interesting part is not whether the answer is right, but how the model explained itself.

In [ ]:
if len(results) == 0:
    print("No results yet. Run Step 7 first.")
else:
    for i, row in enumerate(results, start=1):
        print("=" * 70)
        print("Question", i)
        print(row['question'])
        print()
        print(row['raw_reply'])
        print()
        print("Parsed answer:", row['model_guess'])
        print("Correct answer:", row['correct_answer'])
        print("Correct?:", row['correct'])

## Step 9: Score the model

Small browser models often miss several questions. That is useful evidence, not a failure of the notebook.

In [ ]:
if len(results) == 0:
    print("No results yet. Run Step 7 first.")
else:
    correct_count = sum(1 for row in results if row['correct'])
    print("Score:", correct_count, "out of", len(results))
    print()
    print(results_table[['question', 'correct_answer', 'model_guess', 'correct']])

## Step 10: Optional live question bank from PineSAT

Eric's earlier notebooks pulled from the PineSAT API. In a browser notebook, that may or may not work because the website has to allow cross-origin requests from your JupyterLite page.

This cell tries the live API and tells you clearly what happened.

In [ ]:
LIVE_SECTION = "english"
LIVE_URL = "https://pinesat.com/api/questions?section=" + LIVE_SECTION

try:
    response = await js.fetch(LIVE_URL)
    if not response.ok:
        print("The live API replied with status", response.status)
    else:
        live_questions = (await response.json()).to_py()
        print("Loaded", len(live_questions), "live questions from PineSAT.")
        print("First question record:")
        print(live_questions[0])
except Exception as error:
    print("Could not load the live PineSAT question bank in this browser session.")
    print("Reason:", error)
    print("You can still use the built-in question bank above.")

## Your turn

Try a different section or difficulty. Then switch to another small model and compare the scores. Which mistakes seem random, and which mistakes reveal a real weakness in the model's reasoning or reading?